In [ ]:
import subprocess
import time
import itertools
import os
import pandas as pd
import glob
from datetime import datetime

# ================= 設定區 =================
dic = {
    'batch_size': [1000],
    'epoch': [1000],
    'layer': [4, 3, 2],
    'hidden': [32, 16, 8, 4],
    'data_version': [55], # 確保版本號與訓練腳本一致
    'lr': [0.0003, 0.0001, 0.003, 0.001, 0.03, 0.01],
    'column': ['acceleration_X,acceleration_Y,acceleration_Z', 
               'gyroscope_X,gyroscope_Y,gyroscope_Z', 
               'acceleration_X,acceleration_Y,acceleration_Z,gyroscope_X,gyroscope_Y,gyroscope_Z'],
    'folds': [1, 2, 3, 4, 5],
}

MAX_CONCURRENT_JOBS = 10  # 訓練時的併發數
TRAIN_SCRIPT = "train_GRU_v32.py"
TEST_SCRIPT = "test_GRU_v32.py"
OUTPUT_LOG_DIR = "./test_log"
# =========================================

def get_combinations(params):
    keys = list(params.keys())
    values = list(params.values())
    for combo in itertools.product(*values):
        yield dict(zip(keys, combo))

def run_phase(phase_name, script_name, combinations, max_jobs):
    print(f"\n=== 開始執行階段: {phase_name} ===")
    total_jobs = len(combinations)
    running_processes = []
    
    for i, p in enumerate(combinations):
        # 準備指令
        cmd = [
            'python3', script_name,
            f'--batch_size={p["batch_size"]}',
            f'--epoch={p["epoch"]}',
            f'--layer={p["layer"]}',
            f'--hidden={p["hidden"]}',
            f'--data_version={p["data_version"]}',
            f'--lr={p["lr"]}',
            f'--column={p["column"]}',
            f'--fold={p["folds"]}' # 修正：這裡對應 dic 裡面的 key 'folds'
        ]
        
        # 啟動程序
        proc = subprocess.Popen(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        running_processes.append(proc)
        
        # 進度顯示
        if i % 10 == 0:
            print(f"[{phase_name}] 進度: {i}/{total_jobs} (Running: {len(running_processes)})")

        # 控管併發數
        while len(running_processes) >= max_jobs:
            running_processes = [p for p in running_processes if p.poll() is None]
            if len(running_processes) >= max_jobs:
                time.sleep(1)

    # 等待最後一批完成
    for p in running_processes:
        p.wait()
    print(f"=== {phase_name} 階段完成 ===\n")

def collect_results():
    print(f"=== 正在從 {OUTPUT_LOG_DIR} 彙整最新測試報告 ===")
    
    result_files = glob.glob(os.path.join(OUTPUT_LOG_DIR, "*_result.csv"))
    
    if not result_files:
        print(f"在 {OUTPUT_LOG_DIR} 找不到任何測試結果檔案。")
        return

    all_dfs = []
    for f in result_files:
        try:
            df = pd.read_csv(f)
            if not df.empty:
                all_dfs.append(df)
        except Exception as e:
            print(f"讀取 {f} 失敗: {e}")
    
    if all_dfs:
        final_df = pd.concat(all_dfs, ignore_index=True)
        
        # 修正：根據新的多類別指標排序（使用 macro_f1 排序）
        if "macro_f1" in final_df.columns:
            final_df = final_df.sort_values(by="macro_f1", ascending=False)
        elif "f1" in final_df.columns: # 為了相容舊資料
            final_df = final_df.sort_values(by="f1", ascending=False)
            
        out_name = "Final_Report_3Class_v53.csv"
        final_df.to_csv(out_name, index=False)
        
        print("-" * 50)
        print(f"報告整合完成！共匯總 {len(all_dfs)} 筆測試數據。")
        print(f"最終報告已產出: {out_name}")
        print("-" * 50)
        print("Top 5 最佳模型表現:")
        
        # 修正：印出新版測試結果的欄位
        try:
            print(final_df.head(5)[['run_name', 'acc', 'macro_f1', 'f1_Other', 'f1_Tired', 'f1_notTired']])
        except KeyError:
            # 如果還是混到舊資料，退回印出所有欄位
            print(final_df.head(5))
            
    else:
        print("沒有有效的數據可以合併。")

def main():
    start_time = datetime.now()
    # 將生成器轉換為 list，這行沒問題，這讓你能計算 total_jobs
    combinations = list(get_combinations(dic)) 
    
    # 階段 1: 訓練
    run_phase("Training", TRAIN_SCRIPT, combinations, max_jobs=MAX_CONCURRENT_JOBS)
    
    # 階段 2: 測試
    run_phase("Testing", TEST_SCRIPT, combinations, max_jobs=MAX_CONCURRENT_JOBS)
    
    # 階段 3: 彙整
    collect_results()

    end_time = datetime.now() 
    print(f"總耗時: {end_time - start_time}")

if __name__ == "__main__":
    main()


=== 開始執行階段: Training ===
[Training] 進度: 0/1080 (Running: 1)
